In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

In [2]:
# Load the data
df_sms = pd.read_csv('spam.csv', encoding='latin-1')[['v1', 'v2']]
df_sms.columns = ['label', 'message']

In [3]:
df_sms.head()

,label,message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [4]:
print(df_sms['label'].unique())

['ham' 'spam']


In [5]:
df_sms['label'] = df_sms['label'].str.strip().str.lower()

In [6]:
df_sms['label'] = df_sms['label'].map({
    'ham': 0,
    'spam': 1
})

In [7]:
print(df_sms['label'].isna().sum())

0


In [8]:
import re

df_sms['length'] = df_sms['message'].apply(len)

df_sms['has_url'] = df_sms['message'].apply(
    lambda x: 1 if re.search(r'http[s]?://', x) else 0
)

In [9]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(
    stop_words='english',
    max_features=5000
)

X_text = vectorizer.fit_transform(df_sms['message'])
y_text = df_sms['label']

In [10]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X_text,y_text,test_size=0.2,random_state=42)

In [11]:
from sklearn.linear_model import LogisticRegression
text_model = LogisticRegression(class_weight='balanced',max_iter=1000)

text_model.fit(X_train, y_train)

LogisticRegression(class_weight='balanced', max_iter=1000)

In [12]:
from sklearn.metrics import classification_report

y_pred = text_model.predict(X_test)

print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.98      0.99      0.99       965
           1       0.93      0.90      0.92       150

    accuracy                           0.98      1115
   macro avg       0.96      0.94      0.95      1115
weighted avg       0.98      0.98      0.98      1115



In [13]:
def extract_url_from_text(text):
    urls = re.findall(r'(https?://\S+)', text)
    return urls[0] if urls else None

In [14]:
def remove_urls(text):
    return re.sub(r'http[s]?://\S+', '', text)

In [15]:
from urllib.parse import urlparse

def url_risk_score(url):

    score = 0

    parsed = urlparse(url)
    domain = parsed.netloc.lower()

    # Trusted domains
    trusted_domains = [
        'google.com',
        'github.com',
        'microsoft.com',
        'openai.com',
        'wikipedia.org'
    ]

    if any(td in domain for td in trusted_domains):
        return 0

    suspicious_words = [
        'login',
        'verify',
        'secure',
        'bank',
        'update',
        'free',
        'bonus',
        'win'
    ]

    if len(url) > 75:
        score += 1

    if '@' in url or '%' in url:
        score += 2

    if url.count('-') > 2:
        score += 1

    if not url.startswith('https'):
        score += 1

    for word in suspicious_words:
        if word in url.lower():
            score += 1

    return score

In [16]:
def final_prediction(message):

    # Extract URL first
    url = extract_url_from_text(message)

    # Remove URLs before text classification
    cleaned_message = remove_urls(message)

    # Text prediction
    vectorized = vectorizer.transform([cleaned_message])

    text_pred = text_model.predict(vectorized)[0]

    # URL risk analysis
    risk = 0

    if url:
        risk = url_risk_score(url)

    # Final logic
    if text_pred == 1:
        return "SPAM"

    elif risk >= 3:
        return "SPAM"

    else:
        return "NOT SPAM"

In [17]:
messages = [
    "Hey, are we still meeting today?",

    "Win money now!!! Click http://free-money-login.xyz",

    "Check this out https://recruitment.cdcfib.gov.ng/",

    "URGENT! Verify your account https://secure-bank-update-login.xyz"
]

for msg in messages:
    print(msg)
    print("Prediction:", final_prediction(msg))
    print("-" * 50)

Hey, are we still meeting today?
Prediction: NOT SPAM
--------------------------------------------------
Win money now!!! Click http://free-money-login.xyz
Prediction: SPAM
--------------------------------------------------
Check this out https://recruitment.cdcfib.gov.ng/
Prediction: NOT SPAM
--------------------------------------------------
URGENT! Verify your account https://secure-bank-update-login.xyz
Prediction: SPAM
--------------------------------------------------


In [18]:
import joblib

# Save text classification model
print(joblib.dump(text_model, 'text_model.joblib'))

# Save TF-IDF vectorizer
print(joblib.dump(vectorizer, 'vectorizer.joblib'))

['text_model.joblib']
['vectorizer.joblib']


In [ ]:
from flask import Flask, render_template, request
import joblib
import re
from urllib.parse import urlparse

# =========================
# LOAD MODEL & VECTORIZER
# =========================

text_model = joblib.load('text_model.joblib')
vectorizer = joblib.load('vectorizer.joblib')

# =========================
# INITIALIZE FLASK
# =========================

app = Flask(__name__)

# =========================
# URL EXTRACTION
# =========================

def extract_url_from_text(text):
    urls = re.findall(r'(https?://\S+)', text)
    return urls[0] if urls else None

# =========================
# REMOVE URLS FROM MESSAGE
# =========================

def remove_urls(text):
    return re.sub(r'http[s]?://\S+', '', text)

# =========================
# URL RISK ANALYZER
# =========================

def url_risk_score(url):

    score = 0

    parsed = urlparse(url)
    domain = parsed.netloc.lower()

    trusted_domains = [
        'google.com',
        'github.com',
        'microsoft.com',
        'openai.com',
        'wikipedia.org'
    ]

    # Trusted domains
    if any(td in domain for td in trusted_domains):
        return 0

    suspicious_words = [
        'login',
        'verify',
        'secure',
        'bank',
        'update',
        'free',
        'bonus',
        'win'
    ]

    # Long URL
    if len(url) > 75:
        score += 1

    # Obfuscation characters
    if '@' in url or '%' in url:
        score += 2

    # Too many hyphens
    if url.count('-') > 2:
        score += 1

    # Non-HTTPS
    if not url.startswith('https'):
        score += 1

    # Suspicious keywords
    for word in suspicious_words:
        if word in url.lower():
            score += 1

    return score

# =========================
# FINAL PREDICTION FUNCTION
# =========================

def final_prediction(message):

    # Extract URL
    url = extract_url_from_text(message)

    # Remove URL before text classification
    cleaned_message = remove_urls(message)

    # Vectorize message
    vectorized = vectorizer.transform([cleaned_message])

    # Text prediction
    text_pred = text_model.predict(vectorized)[0]

    # URL risk analysis
    risk = 0

    if url:
        risk = url_risk_score(url)

    # Final decision
    if text_pred == 1:
        return "SPAM"

    elif risk >= 3:
        return "SPAM"

    else:
        return "NOT SPAM"

# =========================
# HOME ROUTE
# =========================

@app.route('/', methods=['GET', 'POST'])
def home():

    prediction = ""

    if request.method == 'POST':

        message = request.form['message']

        prediction = final_prediction(message)

    return render_template('index.html', prediction=prediction)

# =========================
# RUN APPLICATION
# =========================

if __name__ == '__main__':
    app.run(debug=True, use_reloader=False)

 * Serving Flask app '__main__'
 * Debug mode: on


 * Running on http://127.0.0.1:5000
Press CTRL+C to quit
